# Last push: sharp RBF + sqrt raw view

This notebook is deliberately **CV-only**.

- Uses the exact existing 25 folds (`5 seeds × 5 folds`) from the legacy cache.
- Creates **no new CV seeds/folds**.
- Does **not** read validation/test.
- Does **not** create submissions.
- First checks whether the whitened/engineered kernels were under-sharpened.
- Then builds one genuinely new raw view: `sqrt(X) -> foldwise StandardScaler -> RBF`.
- Finally tests a small fixed family of `log_raw + sqrt_raw + engineered` kernels and rank blends.

After it finishes, send the printed tables back to ChatGPT before touching validation/test.


In [1]:
from pathlib import Path
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

import svc_multikernel_experiments as legacy
import svc_final_rescue_experiments as rescue
import svc_last_push_experiments as lastpush

importlib.reload(legacy)
importlib.reload(rescue)
importlib.reload(lastpush)

SEED = 0xFACED
EXISTING_CV_SEEDS = tuple(SEED + i for i in range(5))
N_SPLITS = 5
COORD_N_BINS = 8
COORD_ALPHA = 5.0
PCA_COMPONENTS = 128

SVC_N_JOBS = -1
CACHE_N_JOBS = 1

DATA_DIR = (Path('./data') if not Path('/kaggle/input').exists()
            else Path('/kaggle/input/competitions/week-03-hidden-pairs'))
CACHE_DIR = Path('.svc_multikernel_cache_v2')
RESCUE_CACHE_DIR = Path('.svc_final_rescue_cache_v1')
SQRT_CACHE_DIR = Path('.svc_last_push_sqrt_cache_v1')

train_df = pd.read_csv(DATA_DIR / 'train.csv')
features = [c for c in train_df if c not in {'row_id', 'target'}]
assert len(features) == 512

X_train = train_df[features].to_numpy()
y_train = train_df.target.to_numpy()

assert np.isfinite(X_train).all()
assert np.min(X_train) >= 0

cache = rescue.load_existing_cv_cache(
    y_train,
    EXISTING_CV_SEEDS,
    CACHE_DIR,
    N_SPLITS,
    COORD_N_BINS,
    COORD_ALPHA,
    PCA_COMPONENTS,
)
assert cache.cache_builds == 0
assert len(cache.folds) == 25

rescue_cache = rescue.build_or_load_rescue_cache(
    cache,
    RESCUE_CACHE_DIR,
    rebuild=False,
    n_jobs=CACHE_N_JOBS,
)
assert len(rescue_cache.folds) == 25

print(f'legacy cache hits={cache.cache_hits}, builds={cache.cache_builds}')
print(f'rescue cache hits={rescue_cache.cache_hits}, builds={rescue_cache.cache_builds}')
print('train shape:', X_train.shape)


legacy cache hits=25, builds=0
rescue cache hits=25, builds=0
train shape: (2520, 512)


## 1. Frozen anchors from the current pipeline

In [2]:
FINAL_REPRESENTATIONS = {
    'raw': 'raw_orig',
    'geometry': 'geometry_white16',
    'density': 'density_white5',
    'coord': 'coord_white4',
    'aggregates': 'aggregates_white4',
    'knn': 'knn_orig',
    'pca128': 'pca128_orig',
}

# Current rescue gammas from svc_final_rescue(5).ipynb.
CURRENT_GAMMAS = {
    'raw': 0.002496,
    'geometry': 0.0034171616918486107,
    'density': 0.026340695648809757,
    'coord': 0.02582090408371943,
    'aggregates': 0.0368650778749846,
    'knn': 0.04697254731412588,
    'pca128': 0.0027678574834286906,
}

CURRENT_STANDALONE_CV = {
    'geometry': 0.917002,
    'density': 0.905047,
    'coord': 0.877664,
    'aggregates': 0.839761,
    'knn': 0.882314,
}

K0_WEIGHTS = {
    'raw': 0.68,
    'geometry': 0.10,
    'density': 0.10,
    'aggregates': 0.05,
    'knn': 0.05,
    'coord': 0.02,
    'pca128': 0.00,
}

LEGACY_FINE6_C = 6.0
LEGACY_FINE6_WEIGHTS = {
    'raw': 0.68,
    'geometry': 0.10,
    'aggregates': 0.05,
    'density': 0.10,
    'knn': 0.05,
    'coord': 0.02,
    'pca128': 0.00,
}
LEGACY_FINE6_GAMMAS = {
    'raw': 0.002496,
    'geometry': 0.012772,
    'aggregates': 0.073837,
    'density': 0.023941,
    'knn': 0.187890,
    'coord': 0.055470,
    'pca128': 0.001384,
}

# Historical raw sanity guard.
historical_raw_run = rescue.evaluate_fixed_representation(
    cache, rescue_cache, 'raw_orig', 0.002496, 6.0, SVC_N_JOBS
)
historical_raw_cv = float(historical_raw_run['aucs'].mean())

legacy_fine6_run = legacy.evaluate_config_grid(
    cache,
    {'mode': 'additive', 'weights': LEGACY_FINE6_WEIGHTS},
    LEGACY_FINE6_GAMMAS,
    [LEGACY_FINE6_C],
    SVC_N_JOBS,
)

current_k0_run = rescue.evaluate_additive_candidates(
    cache,
    rescue_cache,
    FINAL_REPRESENTATIONS,
    CURRENT_GAMMAS,
    {'CURRENT_K0': K0_WEIGHTS},
    [3.0],
    SVC_N_JOBS,
)['CURRENT_K0']

anchor_table = pd.DataFrame([
    {
        'model': 'historical_raw',
        'CV_mean': historical_raw_cv,
        'CV_std': historical_raw_run['aucs'].std(),
    },
    {
        'model': 'legacy_fine6',
        'CV_mean': legacy_fine6_run['aucs'].mean(),
        'CV_std': legacy_fine6_run['aucs'].std(),
    },
    {
        'model': 'current_K0',
        'CV_mean': current_k0_run['aucs'].mean(),
        'CV_std': current_k0_run['aucs'].std(),
    },
])
display(anchor_table)

if not (0.927 < historical_raw_cv < 0.931):
    raise RuntimeError(
        f'Historical raw guard failed: got {historical_raw_cv:.9f}; '
        'do not continue until the cache/environment is checked.'
    )


,model,CV_mean,CV_std
0,historical_raw,0.929060,0.007543
1,legacy_fine6,0.932035,0.007212
2,current_K0,0.932150,0.007132


## 2. Sharp-gamma rescue on the active engineered blocks

In [3]:
SHARP_REPRESENTATIONS = {
    'geometry': 'geometry_white16',
    'density': 'density_white5',
    'aggregates': 'aggregates_white4',
    'knn': 'knn_orig',
    'coord': 'coord_white4',
}

SHARP_MULTIPLIERS = [1.0, 2.0, 4.0, 8.0]
SHARP_C_GRID = [3.0, 6.0, 10.0]

sharp_table, sharp_runs = lastpush.sharp_gamma_search(
    cache,
    rescue_cache,
    SHARP_REPRESENTATIONS,
    gamma_multipliers=SHARP_MULTIPLIERS,
    c_grid=SHARP_C_GRID,
    n_jobs=SVC_N_JOBS,
    old_full25=CURRENT_STANDALONE_CV,
)

print('SHARP REPRESENTATION RESULTS')
display(sharp_table)

SHARP_GAMMAS = dict(CURRENT_GAMMAS)
for block, run in sharp_runs.items():
    SHARP_GAMMAS[block] = float(run['gamma'])

sharp_k0_run = rescue.evaluate_additive_candidates(
    cache,
    rescue_cache,
    FINAL_REPRESENTATIONS,
    SHARP_GAMMAS,
    {'SHARP_K0': K0_WEIGHTS},
    [3.0, 6.0],
    SVC_N_JOBS,
)['SHARP_K0']

k0_compare = lastpush.model_table(
    {
        'CURRENT_K0': current_k0_run,
        'SHARP_K0': sharp_k0_run,
    },
    references={
        'fine6': legacy_fine6_run,
        'current_k0': current_k0_run,
    },
)

print('K0 COMPARISON')
display(k0_compare)

USE_SHARP = (
    float(sharp_k0_run['aucs'].mean())
    > float(current_k0_run['aucs'].mean())
)
BASE_GAMMAS = dict(SHARP_GAMMAS if USE_SHARP else CURRENT_GAMMAS)
BASE_K0_RUN = sharp_k0_run if USE_SHARP else current_k0_run

print('USE_SHARP =', USE_SHARP)
print('BASE_GAMMAS =', BASE_GAMMAS)
print('BASE K0 CV =', float(BASE_K0_RUN['aucs'].mean()))


SHARP REPRESENTATION RESULTS


,block,representation,base_gamma,chosen_multiplier,gamma,C,quick5_CV,sharp_full25_CV,sharp_full25_std,old_full25_CV,delta_vs_old
0,density,density_white5,0.105363,1.0,0.105363,3.0,0.900715,0.900675,0.011868,0.905047,-0.004372
1,geometry,geometry_white16,0.027337,1.0,0.027337,3.0,0.898312,0.898403,0.015749,0.917002,-0.018599
2,coord,coord_white4,0.103284,1.0,0.103284,3.0,0.876307,0.872493,0.013094,0.877664,-0.005171
3,knn,knn_orig,0.375780,1.0,0.375780,3.0,0.868326,0.870032,0.012958,0.882314,-0.012282
4,aggregates,aggregates_white4,0.147460,1.0,0.147460,3.0,0.835954,0.835824,0.015652,0.839761,-0.003937


K0 COMPARISON


,model,best_C,CV_mean,CV_std,CV_median,delta_vs_fine6,wins_vs_fine6,losses_vs_fine6,corr_fine6,delta_vs_current_k0,wins_vs_current_k0,losses_vs_current_k0,corr_current_k0
0,CURRENT_K0,3.0,0.932150,0.007132,0.932414,0.000115,13,12,0.991841,0.000000,0,0,1.000000
1,SHARP_K0,3.0,0.931796,0.007406,0.931563,-0.000239,10,15,0.984822,-0.000355,12,13,0.990219


USE_SHARP = False
BASE_GAMMAS = {'raw': 0.002496, 'geometry': 0.0034171616918486107, 'density': 0.026340695648809757, 'coord': 0.02582090408371943, 'aggregates': 0.0368650778749846, 'knn': 0.04697254731412588, 'pca128': 0.0027678574834286906}
BASE K0 CV = 0.9321504157218443


## 3. Build leakage-safe `sqrt_raw` on the exact same folds

This is the only new distance sidecar. For every existing fold:

`original X -> sqrt(X) -> StandardScaler(fit on outer-train only) -> squared Euclidean distances`

No labels are used in this transform.


In [4]:
sqrt_cache = lastpush.build_or_load_sqrt_cache(
    X_train,
    cache,
    SQRT_CACHE_DIR,
    rebuild=False,
    n_jobs=CACHE_N_JOBS,
)

print(
    f"sqrt cache hits={sqrt_cache['cache_hits']}, "
    f"builds={sqrt_cache['cache_builds']}"
)

sqrt_run = lastpush.calibrate_sqrt_raw(
    cache,
    sqrt_cache,
    gamma_multipliers=[1.0, 2.0, 4.0, 8.0],
    c_grid=[1.0, 3.0, 6.0, 10.0],
    n_jobs=SVC_N_JOBS,
)

sqrt_delta = sqrt_run['aucs'] - historical_raw_run['aucs']

sqrt_table = pd.DataFrame([{
    'model': 'sqrt_raw',
    'base_gamma': sqrt_run['base_gamma'],
    'chosen_multiplier': sqrt_run['multiplier'],
    'gamma': sqrt_run['gamma'],
    'C': sqrt_run['C'],
    'quick5_CV': sqrt_run['quick5_CV'],
    'full25_CV': sqrt_run['aucs'].mean(),
    'full25_std': sqrt_run['aucs'].std(),
    'historical_raw_CV': historical_raw_cv,
    'delta_vs_historical_raw': sqrt_delta.mean(),
    'wins_vs_historical_raw': int((sqrt_delta > 0).sum()),
    'losses_vs_historical_raw': int((sqrt_delta < 0).sum()),
}])

print('SQRT RAW RESULT')
display(sqrt_table)


sqrt cache hits=0, builds=25
SQRT RAW RESULT


,model,base_gamma,chosen_multiplier,gamma,C,quick5_CV,full25_CV,full25_std,historical_raw_CV,delta_vs_historical_raw,wins_vs_historical_raw,losses_vs_historical_raw
0,sqrt_raw,0.000686,4.0,0.002745,3.0,0.929567,0.930259,0.007693,0.92906,0.001199,22,3


## 4. Split the 0.68 raw weight between log-raw and sqrt-raw

In [5]:
# Engineered part is frozen at 0.32, exactly as in K0/Fine6.
# Only the 0.68 raw family is split between log1p raw and sqrt raw.
LOG_SQRT_CONFIGS = {
    'S0_LOG': {
        'raw_log': 0.68, 'raw_sqrt': 0.00,
        'geometry': 0.10, 'density': 0.10, 'aggregates': 0.05,
        'knn': 0.05, 'coord': 0.02,
    },
    'S25': {
        'raw_log': 0.51, 'raw_sqrt': 0.17,
        'geometry': 0.10, 'density': 0.10, 'aggregates': 0.05,
        'knn': 0.05, 'coord': 0.02,
    },
    'S50': {
        'raw_log': 0.34, 'raw_sqrt': 0.34,
        'geometry': 0.10, 'density': 0.10, 'aggregates': 0.05,
        'knn': 0.05, 'coord': 0.02,
    },
    'S75': {
        'raw_log': 0.17, 'raw_sqrt': 0.51,
        'geometry': 0.10, 'density': 0.10, 'aggregates': 0.05,
        'knn': 0.05, 'coord': 0.02,
    },
    'S100': {
        'raw_log': 0.00, 'raw_sqrt': 0.68,
        'geometry': 0.10, 'density': 0.10, 'aggregates': 0.05,
        'knn': 0.05, 'coord': 0.02,
    },
}

mixed_runs = lastpush.evaluate_log_sqrt_configs(
    cache,
    rescue_cache,
    sqrt_cache,
    FINAL_REPRESENTATIONS,
    BASE_GAMMAS,
    sqrt_gamma=float(sqrt_run['gamma']),
    configs=LOG_SQRT_CONFIGS,
    c_grid=[3.0, 6.0],
    n_jobs=SVC_N_JOBS,
)

mixed_table = lastpush.model_table(
    mixed_runs,
    references={
        'fine6': legacy_fine6_run,
        'base_k0': BASE_K0_RUN,
    },
)

print('LOG + SQRT MULTI-KERNEL RESULTS')
display(mixed_table)

BEST_MIXED_NAME = str(mixed_table.iloc[0]['model'])
best_mixed_run = mixed_runs[BEST_MIXED_NAME]

print('BEST_MIXED_NAME =', BEST_MIXED_NAME)
print('BEST_MIXED_WEIGHTS =', mixed_runs[BEST_MIXED_NAME]['weights'])
print('BEST_MIXED_C =', mixed_runs[BEST_MIXED_NAME]['best_C'])


LOG + SQRT MULTI-KERNEL RESULTS


,model,best_C,CV_mean,CV_std,CV_median,delta_vs_fine6,wins_vs_fine6,losses_vs_fine6,corr_fine6,delta_vs_base_k0,wins_vs_base_k0,losses_vs_base_k0,corr_base_k0
0,S100,3.0,0.933357,0.007245,0.934193,0.001321,20,5,0.990610,0.001206,24,1,0.998493
1,S75,3.0,0.933100,0.007281,0.933925,0.001065,18,7,0.991226,0.000950,24,1,0.999125
2,S50,3.0,0.932822,0.007256,0.933390,0.000787,17,8,0.991640,0.000672,24,1,0.999603
3,S25,3.0,0.932520,0.007199,0.932870,0.000485,16,9,0.991848,0.000370,24,0,0.999895
4,S0_LOG,3.0,0.932157,0.007129,0.932414,0.000122,13,11,0.991842,0.000007,13,5,1.000000


BEST_MIXED_NAME = S100
BEST_MIXED_WEIGHTS = {'density': 0.1, 'geometry': 0.1, 'knn': 0.05, 'raw_sqrt': 0.68, 'aggregates': 0.05, 'coord': 0.02}
BEST_MIXED_C = 3.0


## 5. Cheap OOF rank-blend check (no validation)

In [6]:
# Recreate the known complementary aux at its frozen settings.
AUX_WEIGHT_CONFIG = {
    'clean_aux': {'pca128': 0.75, 'aggregates': 0.25},
}
clean_aux_run = rescue.evaluate_additive_candidates(
    cache,
    rescue_cache,
    FINAL_REPRESENTATIONS,
    CURRENT_GAMMAS,
    AUX_WEIGHT_CONFIG,
    [30.0],
    SVC_N_JOBS,
)['clean_aux']

blend_model_oof = {
    'legacy_fine6': legacy_fine6_run['oof'],
    'current_k0': current_k0_run['oof'],
    'best_mixed': best_mixed_run['oof'],
    'clean_aux': clean_aux_run['oof'],
}

LAST_PUSH_BLEND_CONFIGS = {
    # Old anchors for direct comparability.
    'OLD_T4': {
        'legacy_fine6': 0.60,
        'current_k0': 0.25,
        'clean_aux': 0.15,
    },
    'OLD_T6': {
        'legacy_fine6': 0.40,
        'current_k0': 0.40,
        'clean_aux': 0.20,
    },

    # Fixed small family around the new strongest kernel.
    'NEW1': {
        'legacy_fine6': 0.20,
        'best_mixed': 0.70,
        'clean_aux': 0.10,
    },
    'NEW2': {
        'legacy_fine6': 0.30,
        'best_mixed': 0.60,
        'clean_aux': 0.10,
    },
    'NEW3': {
        'legacy_fine6': 0.20,
        'best_mixed': 0.65,
        'clean_aux': 0.15,
    },
    'NEW4': {
        'legacy_fine6': 0.15,
        'best_mixed': 0.75,
        'clean_aux': 0.10,
    },
    'NEW5': {
        'best_mixed': 0.80,
        'clean_aux': 0.20,
    },
    'NEW6': {
        'legacy_fine6': 0.25,
        'best_mixed': 0.75,
    },
}

blend_runs = legacy.run_rank_blend_configs(
    cache,
    blend_model_oof,
    LAST_PUSH_BLEND_CONFIGS,
)
blend_table = rescue.blend_result_table(
    cache,
    blend_runs,
    legacy_fine6_run,
)

print('LAST PUSH BLENDS')
display(blend_table)

BEST_BLEND_NAME = str(blend_table.iloc[0]['blend'])
print('BEST_BLEND_NAME =', BEST_BLEND_NAME)
print('BEST_BLEND_WEIGHTS =', LAST_PUSH_BLEND_CONFIGS[BEST_BLEND_NAME])


LAST PUSH BLENDS


,blend,CV_mean,CV_std,CV_median,delta_vs_fine6,wins_vs_fine6,losses_vs_fine6,corr_with_fine6,median_fold_delta,min_fold_delta,max_fold_delta
0,NEW5,0.934295,0.006626,0.934209,0.002259,21,4,0.988663,0.002142,-0.001992,0.005370
1,NEW3,0.934284,0.006729,0.934193,0.002249,23,2,0.992209,0.001858,-0.001394,0.005008
2,NEW4,0.934114,0.006915,0.933949,0.002079,21,4,0.991898,0.002260,-0.001205,0.005031
3,NEW1,0.934111,0.006917,0.934004,0.002076,22,3,0.992586,0.002142,-0.001071,0.004937
4,NEW2,0.934090,0.006887,0.933666,0.002054,23,2,0.993836,0.001992,-0.000811,0.004771
5,OLD_T6,0.933561,0.006495,0.933256,0.001526,20,5,0.994090,0.001724,-0.001354,0.004244
6,NEW6,0.933504,0.007159,0.933359,0.001469,21,4,0.993229,0.001496,-0.001205,0.004590
7,OLD_T4,0.933377,0.006657,0.932839,0.001342,21,4,0.996140,0.001228,-0.000598,0.003370


BEST_BLEND_NAME = NEW5
BEST_BLEND_WEIGHTS = {'best_mixed': 0.8, 'clean_aux': 0.2}


## 6. Copy this summary back to ChatGPT

In [7]:
summary = {
    'historical_raw_CV': float(historical_raw_run['aucs'].mean()),
    'legacy_fine6_CV': float(legacy_fine6_run['aucs'].mean()),
    'current_K0_CV': float(current_k0_run['aucs'].mean()),
    'sharp_K0_CV': float(sharp_k0_run['aucs'].mean()),
    'use_sharp': bool(USE_SHARP),
    'sqrt_raw_CV': float(sqrt_run['aucs'].mean()),
    'sqrt_delta_vs_raw': float(
        sqrt_run['aucs'].mean() - historical_raw_run['aucs'].mean()
    ),
    'sqrt_wins_vs_raw': int(
        (sqrt_run['aucs'] > historical_raw_run['aucs']).sum()
    ),
    'sqrt_losses_vs_raw': int(
        (sqrt_run['aucs'] < historical_raw_run['aucs']).sum()
    ),
    'best_mixed': BEST_MIXED_NAME,
    'best_mixed_CV': float(best_mixed_run['aucs'].mean()),
    'best_mixed_C': float(best_mixed_run['best_C']),
    'best_mixed_weights': mixed_runs[BEST_MIXED_NAME]['weights'],
    'best_blend': BEST_BLEND_NAME,
    'best_blend_CV': float(blend_runs[BEST_BLEND_NAME]['aucs'].mean()),
    'best_blend_weights': LAST_PUSH_BLEND_CONFIGS[BEST_BLEND_NAME],
}

print('\n=== LAST PUSH SUMMARY ===')
for key, value in summary.items():
    print(f'{key}: {value}')

print('\n=== SHARP TABLE ===')
print(sharp_table.to_string(index=False))

print('\n=== K0 COMPARE ===')
print(k0_compare.to_string(index=False))

print('\n=== SQRT TABLE ===')
print(sqrt_table.to_string(index=False))

print('\n=== MIXED TABLE ===')
print(mixed_table.to_string(index=False))

print('\n=== BLEND TABLE ===')
print(blend_table.to_string(index=False))



=== LAST PUSH SUMMARY ===
historical_raw_CV: 0.9290602166792642
legacy_fine6_CV: 0.9320351473922902
current_K0_CV: 0.9321504157218443
sharp_K0_CV: 0.9317957923910305
use_sharp: False
sqrt_raw_CV: 0.9302588813303099
sqrt_delta_vs_raw: 0.0011986646510456778
sqrt_wins_vs_raw: 22
sqrt_losses_vs_raw: 3
best_mixed: S100
best_mixed_CV: 0.9333566389518773
best_mixed_C: 3.0
best_mixed_weights: {'density': 0.1, 'geometry': 0.1, 'knn': 0.05, 'raw_sqrt': 0.68, 'aggregates': 0.05, 'coord': 0.02}
best_blend: NEW5
best_blend_CV: 0.9342945326278661
best_blend_weights: {'best_mixed': 0.8, 'clean_aux': 0.2}

=== SHARP TABLE ===
     block    representation  base_gamma  chosen_multiplier    gamma   C  quick5_CV  sharp_full25_CV  sharp_full25_std  old_full25_CV  delta_vs_old
   density    density_white5    0.105363                1.0 0.105363 3.0   0.900715         0.900675          0.011868       0.905047     -0.004372
  geometry  geometry_white16    0.027337                1.0 0.027337 3.0   0.898312  

In [8]:
# ============================================================
# FINAL NEW5
# FIT: TRAIN ONLY
# PREDICT: TEST
# NO validation.csv is read or used anywhere in this cell
#
# NEW5 =
#   0.80 * S100
# + 0.20 * clean_aux
#
# S100:
#   0.68 sqrt_raw
# + 0.10 geometry_white16
# + 0.10 density_white5
# + 0.05 aggregates_white4
# + 0.05 knn_orig
# + 0.02 coord_white4
#   C = 3
#
# clean_aux:
#   0.75 pca128
# + 0.25 aggregates_white4
#   C = 30
# ============================================================

from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


# ------------------------------------------------------------
# 0. Load TEST only
# ------------------------------------------------------------

test_df = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

assert 'target' not in test_df.columns
assert features == [c for c in test_df.columns if c != 'row_id']

assert len(test_df) == len(sample_submission)

assert np.array_equal(
    test_df['row_id'].to_numpy(),
    sample_submission['row_id'].to_numpy(),
)

X_test = test_df[features].to_numpy()

assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()

assert np.min(X_train) >= 0
assert np.min(X_test) >= 0


# ------------------------------------------------------------
# 1. Freeze EXACTLY the model selected by CV above
# ------------------------------------------------------------

assert BEST_MIXED_NAME == 'S100', (
    f'Expected S100, got {BEST_MIXED_NAME}'
)

assert BEST_BLEND_NAME == 'NEW5', (
    f'Expected NEW5, got {BEST_BLEND_NAME}'
)

assert USE_SHARP is False, (
    'Unexpectedly USE_SHARP=True. '
    'This final cell assumes the executed result where sharp gamma lost.'
)


S100_WEIGHTS = {
    'raw_sqrt': 0.68,
    'geometry': 0.10,
    'density': 0.10,
    'aggregates': 0.05,
    'knn': 0.05,
    'coord': 0.02,
}

S100_C = float(mixed_runs['S100']['best_C'])
SQRT_GAMMA = float(sqrt_run['gamma'])

# For engineered blocks S100 used the non-sharp gammas.
S100_GAMMAS = dict(BASE_GAMMAS)


AUX_WEIGHTS = {
    'pca128': 0.75,
    'aggregates': 0.25,
}

AUX_C = 30.0


assert np.isclose(sum(S100_WEIGHTS.values()), 1.0)
assert np.isclose(sum(AUX_WEIGHTS.values()), 1.0)

assert np.isclose(S100_C, 3.0), S100_C


print('Final architecture frozen.')
print('S100 C:', S100_C)
print('sqrt gamma:', SQRT_GAMMA)
print('S100 weights:', S100_WEIGHTS)
print('aux weights:', AUX_WEIGHTS)
print()


# ------------------------------------------------------------
# 2. Build engineered TRAIN -> TEST views
#
# IMPORTANT:
#   X_train + y_train are the ONLY labeled data here.
#   X_test has no labels and is only transformed.
#
# This recreates the same full-train feature-building path
# that the previous final submission pipeline used.
# ------------------------------------------------------------

print('Building engineered TRAIN -> TEST views...')

final_bundle = rescue.build_external_views(
    X_train,
    y_train,
    X_test,
    builder_seed=SEED + legacy.INNER_SEED_OFFSET,
    n_bins=COORD_N_BINS,
    alpha=COORD_ALPHA,
    pca_components=PCA_COMPONENTS,
)

train_views = final_bundle['train_views']
test_views = final_bundle['external_views']

print('Engineered views ready.')
print()


# ============================================================
# 3. S100 MAIN MODEL
# ============================================================

print('Building S100 kernel...')


# ------------------------------------------------------------
# 3a. sqrt_raw
#
# Fit StandardScaler ONLY on full train.
# Transform test with this frozen scaler.
# ------------------------------------------------------------

sqrt_scaler = StandardScaler()

sqrt_train = sqrt_scaler.fit_transform(
    np.sqrt(X_train)
)

sqrt_test = sqrt_scaler.transform(
    np.sqrt(X_test)
)


Dtr_sqrt = cdist(
    sqrt_train,
    sqrt_train,
    metric='sqeuclidean',
)

Dte_sqrt = cdist(
    sqrt_test,
    sqrt_train,
    metric='sqeuclidean',
)


Ktr_s100 = (
    np.float32(S100_WEIGHTS['raw_sqrt'])
    * np.exp(-SQRT_GAMMA * Dtr_sqrt).astype(np.float32)
)

Kte_s100 = (
    np.float32(S100_WEIGHTS['raw_sqrt'])
    * np.exp(-SQRT_GAMMA * Dte_sqrt).astype(np.float32)
)


# ------------------------------------------------------------
# 3b. Add frozen engineered kernels
# ------------------------------------------------------------

for block in (
    'geometry',
    'density',
    'aggregates',
    'knn',
    'coord',
):
    weight = float(S100_WEIGHTS[block])

    representation = FINAL_REPRESENTATIONS[block]
    gamma = float(S100_GAMMAS[block])

    A = np.asarray(
        train_views[representation],
        dtype=np.float32,
    )

    B = np.asarray(
        test_views[representation],
        dtype=np.float32,
    )

    Dtr = cdist(
        A,
        A,
        metric='sqeuclidean',
    )

    Dte = cdist(
        B,
        A,
        metric='sqeuclidean',
    )

    Ktr_part = np.exp(
        -gamma * Dtr
    ).astype(np.float32)

    Kte_part = np.exp(
        -gamma * Dte
    ).astype(np.float32)

    Ktr_s100 += (
        np.float32(weight) * Ktr_part
    )

    Kte_s100 += (
        np.float32(weight) * Kte_part
    )

    del A, B, Dtr, Dte, Ktr_part, Kte_part


assert Ktr_s100.shape == (
    len(X_train),
    len(X_train),
)

assert Kte_s100.shape == (
    len(X_test),
    len(X_train),
)

assert np.isfinite(Ktr_s100).all()
assert np.isfinite(Kte_s100).all()


# ------------------------------------------------------------
# 3c. Train S100 on FULL TRAIN only
# ------------------------------------------------------------

print('Fitting S100 on train.csv only...')

s100_model = SVC(
    C=S100_C,
    kernel='precomputed',
)

s100_model.fit(
    Ktr_s100,
    y_train,
)

s100_test_score = s100_model.decision_function(
    Kte_s100
)

assert np.isfinite(s100_test_score).all()

print('S100 done.')


# Free the largest temporary objects before aux.
del (
    Dtr_sqrt,
    Dte_sqrt,
    sqrt_train,
    sqrt_test,
    Ktr_s100,
    Kte_s100,
)

print()


# ============================================================
# 4. CLEAN AUX MODEL
#
# 0.75 pca128
# 0.25 aggregates_white4
# C = 30
# ============================================================

print('Building clean_aux kernel...')


Ktr_aux, Kte_aux = rescue._additive_kernel_from_views(
    train_views,
    test_views,
    FINAL_REPRESENTATIONS,
    CURRENT_GAMMAS,
    AUX_WEIGHTS,
)


assert Ktr_aux.shape == (
    len(X_train),
    len(X_train),
)

assert Kte_aux.shape == (
    len(X_test),
    len(X_train),
)

assert np.isfinite(Ktr_aux).all()
assert np.isfinite(Kte_aux).all()


print('Fitting clean_aux on train.csv only...')

aux_model = SVC(
    C=AUX_C,
    kernel='precomputed',
)

aux_model.fit(
    Ktr_aux,
    y_train,
)

aux_test_score = aux_model.decision_function(
    Kte_aux
)

assert np.isfinite(aux_test_score).all()

print('clean_aux done.')

del Ktr_aux, Kte_aux

print()


# ============================================================
# 5. NEW5 PREDICTION-LEVEL RANK BLEND
#
# 80% S100
# 20% clean_aux
# ============================================================

s100_rank = legacy.percentile_rank(
    s100_test_score
)

aux_rank = legacy.percentile_rank(
    aux_test_score
)


new5_raw = (
    0.80 * s100_rank
    + 0.20 * aux_rank
)


# The metric is ROC-AUC, so only ranking matters.
#
# We apply one final percentile-rank transform exactly in the
# spirit of the previous final submission pipeline.
new5_test_score = legacy.percentile_rank(
    new5_raw
)


assert len(new5_test_score) == len(test_df)
assert np.isfinite(new5_test_score).all()

assert np.all(
    (new5_test_score >= 0.0)
    & (new5_test_score <= 1.0)
)


# ============================================================
# 6. READY-TO-SUBMIT CSV
# ============================================================

submission_new5 = sample_submission[
    ['row_id']
].copy()

submission_new5['target'] = np.asarray(
    new5_test_score,
    dtype=float,
)

submission_new5 = submission_new5[
    ['row_id', 'target']
]


assert len(submission_new5) == len(test_df)

assert np.array_equal(
    submission_new5['row_id'].to_numpy(),
    test_df['row_id'].to_numpy(),
)

assert submission_new5['target'].between(
    0.0, 1.0
).all()


SUBMISSION_PATH = Path(
    'submission_NEW5_sqrt_s100_aux20.csv'
)

submission_new5.to_csv(
    SUBMISSION_PATH,
    index=False,
)


# ============================================================
# 7. Final audit
# ============================================================

print()
print('==============================================')
print('FINAL NEW5 SUBMISSION READY')
print('==============================================')

print('Training labeled data : train.csv ONLY')
print('validation.csv used   : NO')
print('test labels used      : NO')

print()

print('Main model:')
print('  S100')
print('  C =', S100_C)
print('  weights =', S100_WEIGHTS)
print('  sqrt gamma =', SQRT_GAMMA)

print()

print('Engineered gammas:')
print({
    k: S100_GAMMAS[k]
    for k in (
        'geometry',
        'density',
        'aggregates',
        'knn',
        'coord',
    )
})

print()

print('Aux model:')
print(
    '  0.75 pca128 + '
    '0.25 aggregates_white4'
)
print('  C =', AUX_C)

print()

print(
    'Final blend: '
    '0.80 * rank(S100) '
    '+ 0.20 * rank(clean_aux)'
)

print()

print(
    'Saved to:',
    SUBMISSION_PATH.resolve(),
)

print('Rows:', len(submission_new5))

print(
    'target min/max:',
    float(submission_new5.target.min()),
    float(submission_new5.target.max()),
)

print(
    'target mean/std:',
    float(submission_new5.target.mean()),
    float(submission_new5.target.std()),
)

display(
    submission_new5.head(10)
)

Final architecture frozen.
S100 C: 3.0
sqrt gamma: 0.002744811743389125
S100 weights: {'raw_sqrt': 0.68, 'geometry': 0.1, 'density': 0.1, 'aggregates': 0.05, 'knn': 0.05, 'coord': 0.02}
aux weights: {'pca128': 0.75, 'aggregates': 0.25}

Building engineered TRAIN -> TEST views...
Engineered views ready.

Building S100 kernel...
Fitting S100 on train.csv only...
S100 done.

Building clean_aux kernel...
Fitting clean_aux on train.csv only...
clean_aux done.


FINAL NEW5 SUBMISSION READY
Training labeled data : train.csv ONLY
validation.csv used   : NO
test labels used      : NO

Main model:
  S100
  C = 3.0
  weights = {'raw_sqrt': 0.68, 'geometry': 0.1, 'density': 0.1, 'aggregates': 0.05, 'knn': 0.05, 'coord': 0.02}
  sqrt gamma = 0.002744811743389125

Engineered gammas:
{'geometry': 0.0034171616918486107, 'density': 0.026340695648809757, 'aggregates': 0.0368650778749846, 'knn': 0.04697254731412588, 'coord': 0.02582090408371943}

Aux model:
  0.75 pca128 + 0.25 aggregates_white4
  C = 30

,row_id,target
0,row_46ad1afad8562239f79c,0.827957
1,row_b42ade9cd57b8d919acb,0.777778
2,row_c951cd5e197b6b4e8b83,0.266129
3,row_a54b3423760d599416ce,0.032706
4,row_a87c87cc47690e2a303f,0.886649
5,row_ed4ebd22745eb750fb85,0.772401
6,row_a9274d52b5b258adc867,0.177419
7,row_ce5ec3e8b85ad36bd3dc,0.209677
8,row_6893a3fde5d281930331,0.082437
9,row_edf1956cd38509ad1888,0.043907
